# Sesión 6 — Delta Lake, versionado, historia y confiabilidad

**Objetivo:** comprender por qué Delta Lake es la base confiable del Lakehouse en Databricks, usando tablas de laboratorio para inspeccionar historia, simular cambios, consultar versiones anteriores, restaurar datos y preparar el camino hacia Gold.

> Mensaje central: SQL nos ayudó a descubrir insights. Delta Lake nos ayuda a confiar en el camino que produjo esos insights.

## Regla de seguridad

En esta sesión **no modificaremos tablas Silver reales**. Todas las operaciones de escritura, actualización o restauración se harán únicamente sobre tablas de laboratorio en:

```text
workspace.delta_lab
```

## 0. Preparación del entorno

Validamos el catálogo `workspace` y los schemas disponibles. Si algún schema no aparece, revisa que las sesiones anteriores se hayan ejecutado correctamente.

## Nota para el instructor — corrección aplicada en Bagazo

El notebook fue ajustado para evitar el error `[DELTA_UNSUPPORTED_MULTI_COL_IN_PREDICATE]`. El punto didáctico es útil: una condición que puede funcionar en un `SELECT` no necesariamente es válida dentro de un `UPDATE` Delta. Para mantener la clase fluida, el `UPDATE` de Bagazo usa condiciones escalares por columna y las validaciones usan `JOIN` contra la vista candidata.


> **Nota instructor — 0. Preparación del entorno**
>
> Mantén el foco en confiabilidad, no en memorizar comandos. Refuerza que Silver es solo lectura durante esta clase y que todos los cambios ocurren en `workspace.delta_lab`.

In [0]:
%sql
USE CATALOG workspace;
SHOW SCHEMAS;

## 1. Retomar la salud de Silver

La tabla de control creada en la Sesión 4 sigue siendo el punto de entrada para hablar de confiabilidad. Recuerda: Delta Lake permite auditar cambios, pero no reemplaza reglas de calidad.

> **Nota instructor — 1. Retomar la salud de Silver**
>
> Mantén el foco en confiabilidad, no en memorizar comandos. Refuerza que Silver es solo lectura durante esta clase y que todos los cambios ocurren en `workspace.delta_lab`.

In [0]:
%sql
SELECT
  dataset,
  tabla,
  capa,
  filas,
  columnas,
  nulos_criticos,
  duplicados_clave,
  reglas_fallidas,
  estado_calidad,
  fecha_validacion,
  observaciones
FROM workspace.control.quality_summary_sesion_04
ORDER BY estado_calidad DESC, dataset, tabla;

In [0]:
%sql
SELECT
  estado_calidad,
  COUNT(*) AS tablas
FROM workspace.control.quality_summary_sesion_04
GROUP BY estado_calidad
ORDER BY estado_calidad;

### Reflexión rápida

`reviews_clean` quedó en estado **REVISAR** por duplicados de clave. Esto no significa que Delta Lake haya fallado; significa que la auditoría técnica y la calidad de negocio son controles complementarios.

## 2. Tablas disponibles para la sesión

Estas tablas se consultan como fuente, pero no se modifican directamente.

> **Nota instructor — 2. Tablas disponibles para la sesión**
>
> Mantén el foco en confiabilidad, no en memorizar comandos. Refuerza que Silver es solo lectura durante esta clase y que todos los cambios ocurren en `workspace.delta_lab`.

In [0]:
%sql
SHOW TABLES IN workspace.lumi_silver;
SHOW TABLES IN workspace.bagazo_silver;
SHOW TABLES IN workspace.control;

## 3. Inspeccionar metadata de tablas Silver

`DESCRIBE DETAIL` permite revisar información técnica de una tabla Delta: formato, ubicación, tamaño, propiedades y otra metadata disponible.

> **Nota instructor — 3. Inspeccionar metadata de tablas Silver**
>
> Mantén el foco en confiabilidad, no en memorizar comandos. Refuerza que Silver es solo lectura durante esta clase y que todos los cambios ocurren en `workspace.delta_lab`.

In [0]:
%sql
DESCRIBE DETAIL workspace.lumi_silver.orders_clean;

In [0]:
%sql
DESCRIBE DETAIL workspace.bagazo_silver.operacion_ingenios_clean;

## 4. Revisar historia de tablas Silver

Podemos consultar la historia de una tabla Silver. **Solo la revisamos; no la modificamos.**

> **Nota instructor — 4. Revisar historia de tablas Silver**
>
> Mantén el foco en confiabilidad, no en memorizar comandos. Refuerza que Silver es solo lectura durante esta clase y que todos los cambios ocurren en `workspace.delta_lab`.

In [0]:
%sql
DESCRIBE HISTORY workspace.lumi_silver.orders_clean;

In [0]:
%sql
DESCRIBE HISTORY workspace.bagazo_silver.operacion_ingenios_clean;

## 5. Crear el schema de laboratorio

A partir de aquí trabajaremos en una caja de arena segura.

> **Nota instructor — 5. Crear el schema de laboratorio**
>
> Mantén el foco en confiabilidad, no en memorizar comandos. Refuerza que Silver es solo lectura durante esta clase y que todos los cambios ocurren en `workspace.delta_lab`.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.delta_lab;
SHOW TABLES IN workspace.delta_lab;

## 6. Crear tablas de laboratorio

Para garantizar reproducibilidad, eliminamos y recreamos únicamente las tablas lab. Esto permite que `VERSION AS OF 0` sea la versión inicial de esta ejecución.

> Seguridad: estas operaciones solo afectan `workspace.delta_lab`.

> **Nota instructor — 6. Crear tablas de laboratorio**
>
> Mantén el foco en confiabilidad, no en memorizar comandos. Refuerza que Silver es solo lectura durante esta clase y que todos los cambios ocurren en `workspace.delta_lab`.

In [0]:
%sql
DROP TABLE IF EXISTS workspace.delta_lab.orders_reliability_lab;

CREATE TABLE workspace.delta_lab.orders_reliability_lab AS
SELECT *
FROM workspace.lumi_silver.orders_clean
WHERE order_status = 'delivered'
LIMIT 5000;

In [0]:
%sql
DROP TABLE IF EXISTS workspace.delta_lab.bagazo_reliability_lab;

CREATE TABLE workspace.delta_lab.bagazo_reliability_lab AS
SELECT *
FROM workspace.bagazo_silver.operacion_ingenios_clean;

## 7. Validar tablas lab

Confirmamos que las tablas lab existen y tienen datos.

> **Nota instructor — 7. Validar tablas lab**
>
> Mantén el foco en confiabilidad, no en memorizar comandos. Refuerza que Silver es solo lectura durante esta clase y que todos los cambios ocurren en `workspace.delta_lab`.

In [0]:
%sql
SELECT 'orders_reliability_lab' AS tabla, COUNT(*) AS registros
FROM workspace.delta_lab.orders_reliability_lab
UNION ALL
SELECT 'bagazo_reliability_lab' AS tabla, COUNT(*) AS registros
FROM workspace.delta_lab.bagazo_reliability_lab;

In [0]:
%sql
DESCRIBE HISTORY workspace.delta_lab.orders_reliability_lab;
DESCRIBE HISTORY workspace.delta_lab.bagazo_reliability_lab;

## 8. Laboratorio Lumi — seleccionar una orden candidata

Seleccionaremos una orden de laboratorio para simular un cambio controlado. La vista temporal evita copiar manualmente un `order_id`.

> **Nota instructor — 8. Laboratorio Lumi — seleccionar una orden candidata**
>
> Mantén el foco en confiabilidad, no en memorizar comandos. Refuerza que Silver es solo lectura durante esta clase y que todos los cambios ocurren en `workspace.delta_lab`.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v_order_lab_candidate AS
SELECT order_id
FROM workspace.delta_lab.orders_reliability_lab
ORDER BY order_id
LIMIT 1;

SELECT * FROM v_order_lab_candidate;

In [0]:
%sql
SELECT order_id, order_status, delay_days, is_late
FROM workspace.delta_lab.orders_reliability_lab
WHERE order_id = (SELECT order_id FROM v_order_lab_candidate);

## 9. Laboratorio Lumi — simular cambio controlado

Vamos a modificar `order_status` en una sola orden de laboratorio. Esta operación debe generar una nueva versión.

> Antes de ejecutar: verifica que la tabla pertenece a `workspace.delta_lab`.

> **Nota instructor — 9. Laboratorio Lumi — simular cambio controlado**
>
> Mantén el foco en confiabilidad, no en memorizar comandos. Refuerza que Silver es solo lectura durante esta clase y que todos los cambios ocurren en `workspace.delta_lab`.

In [0]:
%sql
UPDATE workspace.delta_lab.orders_reliability_lab
SET order_status = 'canceled'
WHERE order_id = (SELECT order_id FROM v_order_lab_candidate);

In [0]:
%sql
DESCRIBE HISTORY workspace.delta_lab.orders_reliability_lab;

## 10. Time Travel — consultar versión anterior

Ahora consultaremos la versión inicial de la tabla lab. Como recreamos la tabla en este notebook, normalmente la versión inicial es `0`.

**TODO 1:** si tu historial muestra otra versión inicial por una ejecución anterior, cambia `VERSION AS OF 0` por la versión correcta.

> **Nota instructor — 10. Time Travel — consultar versión anterior**
>
> Mantén el foco en confiabilidad, no en memorizar comandos. Refuerza que Silver es solo lectura durante esta clase y que todos los cambios ocurren en `workspace.delta_lab`.

In [0]:
%sql
SELECT order_id, order_status, delay_days, is_late
FROM workspace.delta_lab.orders_reliability_lab VERSION AS OF 0
WHERE order_id = (SELECT order_id FROM v_order_lab_candidate);

In [0]:
%sql
SELECT order_id, order_status, delay_days, is_late
FROM workspace.delta_lab.orders_reliability_lab
WHERE order_id = (SELECT order_id FROM v_order_lab_candidate);

## 11. Comparar versión actual vs versión anterior

Esta consulta muestra la diferencia entre el estado inicial y el estado actual de la orden.

> **Nota instructor — 11. Comparar versión actual vs versión anterior**
>
> Mantén el foco en confiabilidad, no en memorizar comandos. Refuerza que Silver es solo lectura durante esta clase y que todos los cambios ocurren en `workspace.delta_lab`.

In [0]:
%sql
WITH version_inicial AS (
  SELECT
    order_id,
    order_status AS order_status_version_0,
    delay_days AS delay_days_version_0
  FROM workspace.delta_lab.orders_reliability_lab VERSION AS OF 0
  WHERE order_id = (SELECT order_id FROM v_order_lab_candidate)
), version_actual AS (
  SELECT
    order_id,
    order_status AS order_status_actual,
    delay_days AS delay_days_actual
  FROM workspace.delta_lab.orders_reliability_lab
  WHERE order_id = (SELECT order_id FROM v_order_lab_candidate)
)
SELECT
  a.order_id,
  i.order_status_version_0,
  a.order_status_actual,
  i.delay_days_version_0,
  a.delay_days_actual
FROM version_actual a
JOIN version_inicial i
  ON a.order_id = i.order_id;

## 12. Restore controlado de Lumi lab

Restauramos la tabla lab a la versión inicial. Recuerda: `RESTORE` también queda registrado en la historia.

> **Nota instructor — 12. Restore controlado de Lumi lab**
>
> Mantén el foco en confiabilidad, no en memorizar comandos. Refuerza que Silver es solo lectura durante esta clase y que todos los cambios ocurren en `workspace.delta_lab`.

In [0]:
%sql
RESTORE TABLE workspace.delta_lab.orders_reliability_lab TO VERSION AS OF 0;

In [0]:
%sql
DESCRIBE HISTORY workspace.delta_lab.orders_reliability_lab;

In [0]:
%sql
SELECT order_id, order_status, delay_days, is_late
FROM workspace.delta_lab.orders_reliability_lab
WHERE order_id = (SELECT order_id FROM v_order_lab_candidate);

## 13. Laboratorio Bagazo — simular una lectura inválida

Ahora trabajaremos con el caso espejo empresarial. Simularemos una lectura de lluvia negativa para mostrar que Delta Lake audita el cambio, pero las reglas de calidad son las que detectan el problema.

> **Nota instructor — 13. Laboratorio Bagazo — simular una lectura inválida**
>
> Mantén el foco en confiabilidad, no en memorizar comandos. Refuerza que Silver es solo lectura durante esta clase y que todos los cambios ocurren en `workspace.delta_lab`.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v_bagazo_lab_candidate AS
SELECT fecha, ingenio
FROM workspace.delta_lab.bagazo_reliability_lab
WHERE ingenio = 'Providencia'
ORDER BY fecha
LIMIT 1;

SELECT * FROM v_bagazo_lab_candidate;

In [0]:
%sql
-- Corrección: evitamos predicados IN multicolumna en operaciones Delta.
SELECT t.fecha, t.ingenio, t.lluvia_mm, t.cana_molida_ton, t.bagazo_entregado_ton
FROM workspace.delta_lab.bagazo_reliability_lab AS t
INNER JOIN v_bagazo_lab_candidate AS c
  ON t.fecha = c.fecha
 AND t.ingenio = c.ingenio;

## 14. Ejecutar el cambio inválido en Bagazo lab

> Seguridad: la tabla sigue siendo `workspace.delta_lab.bagazo_reliability_lab`.

> **Nota instructor — 14. Ejecutar el cambio inválido en Bagazo lab**
>
> Mantén el foco en confiabilidad, no en memorizar comandos. Refuerza que Silver es solo lectura durante esta clase y que todos los cambios ocurren en `workspace.delta_lab`.

In [0]:
%sql
-- Seguridad: esta operación modifica SOLO la tabla de laboratorio.
-- Corrección aplicada: Delta no soporta IN multicolumna dentro de UPDATE.
UPDATE workspace.delta_lab.bagazo_reliability_lab
SET lluvia_mm = -10
WHERE fecha = (SELECT fecha FROM v_bagazo_lab_candidate)
  AND ingenio = (SELECT ingenio FROM v_bagazo_lab_candidate);

In [0]:
%sql
SELECT fecha, ingenio, lluvia_mm, cana_molida_ton, bagazo_entregado_ton
FROM workspace.delta_lab.bagazo_reliability_lab
WHERE lluvia_mm < 0;

In [0]:
%sql
DESCRIBE HISTORY workspace.delta_lab.bagazo_reliability_lab;

## 15. Time Travel y restore en Bagazo

**TODO 2:** revisa el historial y confirma cuál es la versión inicial antes de ejecutar el restore. En una ejecución limpia será `0`.

> **Nota instructor — 15. Time Travel y restore en Bagazo**
>
> Mantén el foco en confiabilidad, no en memorizar comandos. Refuerza que Silver es solo lectura durante esta clase y que todos los cambios ocurren en `workspace.delta_lab`.

In [0]:
%sql
-- Time Travel seguro sobre la versión inicial de la tabla lab.
WITH bagazo_version_inicial AS (
  SELECT fecha, ingenio, lluvia_mm, cana_molida_ton, bagazo_entregado_ton
  FROM workspace.delta_lab.bagazo_reliability_lab VERSION AS OF 0
)
SELECT t.fecha, t.ingenio, t.lluvia_mm, t.cana_molida_ton, t.bagazo_entregado_ton
FROM bagazo_version_inicial AS t
INNER JOIN v_bagazo_lab_candidate AS c
  ON t.fecha = c.fecha
 AND t.ingenio = c.ingenio;

In [0]:
%sql
RESTORE TABLE workspace.delta_lab.bagazo_reliability_lab TO VERSION AS OF 0;

In [0]:
%sql
-- Corrección: evitamos predicados IN multicolumna en operaciones Delta.
SELECT t.fecha, t.ingenio, t.lluvia_mm, t.cana_molida_ton, t.bagazo_entregado_ton
FROM workspace.delta_lab.bagazo_reliability_lab AS t
INNER JOIN v_bagazo_lab_candidate AS c
  ON t.fecha = c.fecha
 AND t.ingenio = c.ingenio;

## 16. Crear resumen de confiabilidad de la sesión

Esta tabla opcional resume la revisión de confiabilidad realizada antes de construir Gold en la Sesión 7.

> **Nota instructor — 16. Crear resumen de confiabilidad de la sesión**
>
> Mantén el foco en confiabilidad, no en memorizar comandos. Refuerza que Silver es solo lectura durante esta clase y que todos los cambios ocurren en `workspace.delta_lab`.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.control.delta_reliability_summary_sesion_06 AS
SELECT
  'workspace.delta_lab.orders_reliability_lab' AS tabla,
  'Lumi: laboratorio de versionado sobre pedidos' AS proposito,
  'DESCRIBE HISTORY + VERSION AS OF + RESTORE' AS evidencia_usada,
  'REVISADA' AS estado_revision,
  'No modifica Silver real. Lista como práctica de auditoría previa a Gold.' AS observaciones,
  current_timestamp() AS fecha_revision
UNION ALL
SELECT
  'workspace.delta_lab.bagazo_reliability_lab' AS tabla,
  'Bagazo: laboratorio de trazabilidad operativa' AS proposito,
  'DESCRIBE HISTORY + detección de rango inválido + RESTORE' AS evidencia_usada,
  'REVISADA' AS estado_revision,
  'No modifica Silver real. Útil para explicar calidad + confiabilidad.' AS observaciones,
  current_timestamp() AS fecha_revision;

SELECT * FROM workspace.control.delta_reliability_summary_sesion_06;

## 17. Checklist de confiabilidad pre-Gold

Completa este checklist antes de pasar a la Sesión 7.

**TODO 3:** completa la conclusión ejecutiva.

```text
Tabla:
Propósito analítico:
Última versión revisada:
Operaciones recientes:
Riesgos detectados:
¿Puede alimentar Gold?: Sí / No / Con observaciones
Evidencia SQL usada:
Recomendación:
```

### Conclusión ejecutiva

Escribe aquí tu conclusión:

> ...

> **Nota instructor — 17. Checklist de confiabilidad pre-Gold**
>
> Mantén el foco en confiabilidad, no en memorizar comandos. Refuerza que Silver es solo lectura durante esta clase y que todos los cambios ocurren en `workspace.delta_lab`.

# Retos finales

Los retos no interrumpen el flujo principal. Úsalos como práctica de cierre o trabajo autónomo.

## Reto Nivel 1 — Historia de tabla

Elige una tabla lab, ejecuta `DESCRIBE HISTORY` e identifica:

- versión inicial;
- última versión;
- última operación;
- evidencia más importante.

## Reto Nivel 2 — Time Travel y comparación

Consulta una versión anterior con `VERSION AS OF`, compárala contra la versión actual y explica qué cambió.

## Reto consultor — Checklist pre-Gold

Entrega el checklist de confiabilidad para una tabla que podría alimentar Gold en la Sesión 7.

# Soluciones de retos — Instructor

## Solución Reto Nivel 1 — Historia de tabla

In [0]:
%sql
DESCRIBE HISTORY workspace.delta_lab.orders_reliability_lab;

Lectura esperada:

- La versión 0 corresponde a la creación inicial de la tabla lab.
- Las versiones posteriores muestran operaciones como UPDATE y RESTORE.
- La evidencia más útil es `operation`, `timestamp`, `userName` y `operationMetrics`.

## Solución Reto Nivel 2 — Time Travel y comparación

In [0]:
%sql
WITH actual AS (
  SELECT order_id, order_status, delay_days
  FROM workspace.delta_lab.orders_reliability_lab
  WHERE order_id = (SELECT order_id FROM v_order_lab_candidate)
), inicial AS (
  SELECT order_id, order_status, delay_days
  FROM workspace.delta_lab.orders_reliability_lab VERSION AS OF 0
  WHERE order_id = (SELECT order_id FROM v_order_lab_candidate)
)
SELECT
  a.order_id,
  i.order_status AS order_status_inicial,
  a.order_status AS order_status_actual,
  i.delay_days AS delay_days_inicial,
  a.delay_days AS delay_days_actual
FROM actual a
JOIN inicial i USING(order_id);

## Solución Reto consultor — Checklist pre-Gold

```text
Tabla: workspace.lumi_silver.orders_clean
Propósito analítico: alimentar KPIs de pedidos, entregas, demoras y experiencia de cliente.
Última versión revisada: revisar salida de DESCRIBE HISTORY.
Operaciones recientes: creación/carga Silver; sin modificaciones manuales en esta sesión.
Riesgos detectados: validar consistencia con payments, order_items y reviews antes de Gold.
¿Puede alimentar Gold?: Sí, con revisión de granularidad en joins.
Evidencia SQL usada: DESCRIBE DETAIL, DESCRIBE HISTORY y quality_summary_sesion_04.
Recomendación: usar como fuente de fact_orders/fact_delivery en Sesión 7, evitando joins que dupliquen pagos o reviews.
```

# Guion adicional para explicar Free Edition vs Azure Databricks empresarial

En Free Edition trabajamos con notebooks, SQL y tablas Delta administradas. En Azure Databricks empresarial se agregaría una capa más robusta de gobierno: Unity Catalog con permisos finos, separación dev/test/prod, lineage, auditoría, workflows productivos, integración con ADLS Gen2 y CI/CD con Azure DevOps o GitHub Actions.